<a href="https://colab.research.google.com/github/anujithesh26-del/SIH-2026-AI-ML-ENABLED-SEAMLESS-NAVIGATION/blob/TASK--A-IO-VNB-DATASET-UNDERSTANDING-AND-PREPROCESSING/SIH_STEP_6%2B7_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import google.colab.files as files

# The input CSV file will be uploaded by the user.
# The output directory will be the current working directory for download.

WINDOW_LENGTH_S = 5.0
STRIDE_S = 2.5
SESSION_ID = "VW10"
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # test = remainder

exclude_cols = {
    "time_s", "s_time_corrected_s", "Sample period (seconds)",
    "S_GPS SATELLITES IN RANGE", "S_DATE (YYYY-MO-DD HH-MI-SS_SSS)",
    "S_TIME SINCE START (ms)", "Time Since Start of Day (seconds)",
}

print("Please upload the cleaned_MERGED_SVW10-VW10.csv file.")
uploaded = files.upload()

# Assuming only one file is uploaded, get its name
INPUT_CSV_FILENAME = next(iter(uploaded))
df = pd.read_csv(INPUT_CSV_FILENAME).sort_values("time_s").reset_index(drop=True)
dt = df["time_s"].diff().dropna()
sample_rate_hz = round(1.0 / dt.median(), 4)
window_len_samples = int(round(WINDOW_LENGTH_S * sample_rate_hz))
stride_samples = int(round(STRIDE_S * sample_rate_hz))
feature_cols = [c for c in df.columns if c not in exclude_cols]

n_rows = len(df)
n_train_rows = int(round(n_rows * TRAIN_FRAC))
n_val_rows = int(round(n_rows * VAL_FRAC))
segments = {
    "train": df.iloc[0:n_train_rows].reset_index(drop=True),
    "val":   df.iloc[n_train_rows:n_train_rows + n_val_rows].reset_index(drop=True),
    "test":  df.iloc[n_train_rows + n_val_rows:n_rows].reset_index(drop=True),
}

def window_segment(seg_df, split_name):
    X, starts, ends, dropout = [], [], [], []
    i = 0
    while i + window_len_samples <= len(seg_df):
        w = seg_df.iloc[i:i + window_len_samples]
        X.append(w[feature_cols].to_numpy(dtype=np.float32))
        starts.append(w["time_s"].iloc[0])
        ends.append(w["time_s"].iloc[-1])
        dropout.append(bool(w["No of GPS Satellites Available"].isna().any()))
        i += stride_samples
    n = len(X)
    print(f"{split_name}: {len(seg_df)} rows -> {n} windows "
          f"({starts[0]:.1f}s-{ends[-1]:.1f}s)" if n else f"{split_name}: 0 windows")
    return (np.stack(X) if n else np.empty((0, window_len_samples, len(feature_cols)), dtype=np.float32),
            np.array(starts), np.array(ends), np.array(dropout, dtype=bool))

all_X, all_start, all_end, all_dropout, all_split = [], [], [], [], []
for name, seg in segments.items():
    X, s, e, drop = window_segment(seg, name)
    all_X.append(X); all_start.append(s); all_end.append(e); all_dropout.append(drop)
    all_split.extend([name] * len(X))

X = np.concatenate(all_X, axis=0)
window_start_s = np.concatenate(all_start)
window_end_s = np.concatenate(all_end)
gps_dropout_flag = np.concatenate(all_dropout)
split = np.array(all_split)
session_id = np.array([SESSION_ID] * len(X))

print(f"\nTotal windows: {len(X)}  (shape per window: {X.shape[1:]})")
for name in ["train", "val", "test"]:
    mask = split == name
    print(f"{name}: {mask.sum()} windows, {gps_dropout_flag[mask].sum()} GPS-dropout-flagged")

# confirm no time overlap across split boundaries
train_end = window_end_s[split == "train"].max()
val_start = window_start_s[split == "val"].min()
val_end = window_end_s[split == "val"].max()
test_start = window_start_s[split == "test"].min()
assert train_end <= val_start, f"LEAK: train ends {train_end}, val starts {val_start}"
assert val_end <= test_start, f"LEAK: val ends {val_end}, test starts {test_start}"
print("\nNo train/val/test time overlap confirmed.")

# Save the output NPZ file to the current directory and download it.
out_npz_filename = "vw10_windows_split.npz"
np.savez(
    out_npz_filename, X=X, feature_names=np.array(feature_cols), session_id=session_id,
    window_start_s=window_start_s, window_end_s=window_end_s,
    gps_dropout_flag=gps_dropout_flag, split=split,
    sample_rate_hz=np.array([sample_rate_hz]), window_length_s=np.array([WINDOW_LENGTH_S]),
    stride_s=np.array([STRIDE_S]),
    split_mode=np.array(["placeholder_time_contiguous_single_session_split_then_window"]),
)
print(f"Saved: {out_npz_filename}")

# Download the generated file
files.download(out_npz_filename)

Please upload the cleaned_MERGED_SVW10-VW10.csv file.


Saving cleaned_MERGED SVW10-VW10.csv to cleaned_MERGED SVW10-VW10.csv
train: 456 rows -> 17 windows (58062.4s-58107.3s)
val: 98 rows -> 2 windows (58108.0s-58115.4s)
test: 98 rows -> 2 windows (58117.8s-58125.2s)

Total windows: 21  (shape per window: (50, 48))
train: 17 windows, 0 GPS-dropout-flagged
val: 2 windows, 2 GPS-dropout-flagged
test: 2 windows, 2 GPS-dropout-flagged

No train/val/test time overlap confirmed.
Saved: vw10_windows_split.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>